# Cross-domain engineering SVG patch training

Select Kernel > Colab > Auto Connect with a **GPU** runtime, then Run All.

Trains a LoRA on 8,000 lineage-disjoint SVG edit examples across five domains (building plans,
furniture, mechanical parts, DC circuits, water piping) and scores 100 held-out examples before and
after. The target is constrained patch JSON, not a redrawn SVG, so the model never performs
coordinate arithmetic.

Every gold patch was applied to its source and proved to reproduce the target tree: 10,000 of 10,000.
Checkpoints go to Drive, so a recycled runtime resumes instead of restarting.

In [1]:
import subprocess, sys, torch

def run(args, **kw):
    print('RUN:', ' '.join(map(str, args)), flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, **kw)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError('command failed: ' + ' '.join(map(str, args)))

assert torch.cuda.is_available(), 'Connect a GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


In [2]:
# Fetch from GitHub rather than uploading: files.upload() is a browser-only widget that the VS Code
# notebook renderer disables, which is what stalled the previous attempt.
import pathlib
R = '/content/SVG'
if pathlib.Path(R, '.git').exists():
    run(['git', '-C', R, 'fetch', '-q', '--depth', '1', 'origin', 'cvpr2027-research-package'])
    run(['git', '-C', R, 'reset', '-q', '--hard', 'FETCH_HEAD'])
else:
    run(['git', 'clone', '-q', '--depth', '1', '-b', 'cvpr2027-research-package',
         '--filter=blob:none', '--sparse', 'https://github.com/ayushdebnath012/SVG.git', R])
run(['git', '-C', R, 'sparse-checkout', 'set', 'cvpr2027/scripts', 'cvpr2027/src',
     'cvpr2027/data/engsvg-crossdomain-edit-v1'])
run(['git', '-C', R, 'log', '--oneline', '-1'])

RUN: git clone -q --depth 1 -b cvpr2027-research-package --filter=blob:none --sparse https://github.com/ayushdebnath012/SVG.git /content/SVG
RUN: git -C /content/SVG sparse-checkout set cvpr2027/scripts cvpr2027/src cvpr2027/data/engsvg-crossdomain-edit-v1
RUN: git -C /content/SVG log --oneline -1
2cce690 Size the evaluation for a T4


In [3]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.48', 'peft>=0.14',
     'accelerate>=1.2', 'svgpathtools==1.7.2'])
# Colab preinstalls torchao 0.10 and peft's LoRA dispatch raises on anything below 0.16.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)

RUN: /usr/bin/python3 -m pip install -q transformers>=4.48 peft>=0.14 accelerate>=1.2 svgpathtools==1.7.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.4 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], returncode=0)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
project = '/content/SVG/cvpr2027'
env = dict(os.environ, PYTHONPATH='src:scripts')
run([sys.executable, 'scripts/train_crossdomain_svg_patcher.py',
     '--data', 'data/engsvg-crossdomain-edit-v1',
     '--out', '/content/drive/MyDrive/engsvg-crossdomain-run',
     '--epochs', '1', '--eval-count', '30',
     '--max-length', '4096', '--max-new-tokens', '1400'], cwd=project, env=env)

RUN: /usr/bin/python3 scripts/train_crossdomain_svg_patcher.py --data data/engsvg-crossdomain-edit-v1 --out /content/drive/MyDrive/engsvg-crossdomain-run --epochs 1 --eval-count 30 --max-length 4096 --max-new-tokens 1400


In [ ]:
import json, pathlib
s = json.loads(pathlib.Path('/content/drive/MyDrive/engsvg-crossdomain-run/summary.json').read_text())
print(json.dumps(s, indent=2)[:4000])